# A01 SETUP

In [0]:
# Import PySpark SQL functions and Window for analytical queries
from pyspark.sql import functions as F, Window
# reduce lets us union multiple DataFrames iteratively
from functools import reduce
# re for regex validation; uuid for unique run identifier
import re, uuid

# Catalog name for all schemas in this pipeline
CAT = 'workspace'
# Validate catalog name is a safe SQL identifier (letters, digits, underscore)
assert re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', CAT)

# Generate a unique run ID to tag every row produced in this pipeline run
RUN = str(uuid.uuid4())

# Define the five schema layers: raw landing, bronze (as-read), silver (typed), gold (reporting), audit/QA
RAW = f'{CAT}.olist_raw11'   # Raw volume storage schema
B   = f'{CAT}.olist_bronze11' # Bronze: raw CSV as-is
S   = f'{CAT}.olist_silver11' # Silver: typed and cleaned
G   = f'{CAT}.olist_gold11'   # Gold: dimensional model and facts
Q   = f'{CAT}.olist_audit11'  # Audit: QA checks, profiles, diagnostics

# Create all five schemas if they don't already exist
for schema in [RAW, B, S, G, Q]:
    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema}')

# Create a UC volume to hold uploaded CSV source files
spark.sql(f'CREATE VOLUME IF NOT EXISTS {RAW}.files')

# Build the base path and input directory for the uploaded source files
BASE = f'/Volumes/{CAT}/olist_raw11/files'
INPUT = f'{BASE}/input/release_01'
dbutils.fs.mkdirs(INPUT)

# Helper: overwrite a Delta table by name (used throughout the notebook)
def save(df, name):
    df.write.format('delta').mode('overwrite').saveAsTable(name)

# Global list to accumulate QA check results for the current run
qa = []

# Helper: run a QA check, log it to the audit table, and halt if blocking failures exist
def check(name, failures, blocking=True):
    failures = int(failures)
    # Append the check result to the in-memory list
    qa.append((RUN, name, failures, bool(blocking)))
    # Persist the full QA log to the audit.checks table (overwritten each call)
    save(spark.createDataFrame(qa,
        'run_id STRING, test STRING, failures LONG, blocking BOOLEAN'),
        f'{Q}.checks')
    print(name, failures)
    # If the check is blocking and has failures, raise an AssertionError to stop the pipeline
    assert not (blocking and failures), f'STOP: {name}'

print('SETUP OK', INPUT)


SETUP OK /Volumes/workspace/olist_raw11/files/input/release_01


# A02 Read the nine source files

In [0]:
# Map logical table names to the nine source CSV filenames from the Olist dataset
files = {
 'orders':'olist_orders_dataset.csv',           # Order header data
 'items':'olist_order_items_dataset.csv',        # Line items per order
 'payments':'olist_order_payments_dataset.csv',   # Payment records per order
 'reviews':'olist_order_reviews_dataset.csv',     # Customer review scores
 'customers':'olist_customers_dataset.csv',       # Customer master records
 'products':'olist_products_dataset.csv',         # Product master records
 'sellers':'olist_sellers_dataset.csv',           # Seller master records
 'geolocation':'olist_geolocation_dataset.csv',   # ZIP-to-lat/lng lookup
 'translation':'product_category_name_translation.csv'}  # PT->EN category names

# List all files actually present in the input directory
present = {x.name for x in dbutils.fs.ls(INPUT)}
# Assert every expected CSV was uploaded; fail immediately if any are missing
assert set(files.values()) <= present, 'Upload all nine named CSV files'

# Build a manifest of (run_id, table_name, file, row_count) for each loaded file
manifest = []
for name, file in files.items():
    # Read CSV with header, no schema inference (all strings), strict UTF-8, quoted multiline fields
    df = (spark.read.option('header', True)
        .option('inferSchema', False).option('encoding', 'UTF-8')
        .option('multiLine', True).option('quote', '"')
        .option('escape', '"').option('mode', 'FAILFAST')
        .csv(f'{INPUT}/{file}'))
    # Sanity check: if only one column, the CSV separator is likely wrong
    assert len(df.columns) > 1, f'Check CSV separator: {file}'
    # Count rows and verify the file is not empty
    n = df.count()
    assert n > 0, f'Empty file: {file}'
    # Tag every row with source filename and run ID for traceability
    df = (df.withColumn('_source', F.lit(file))
        .withColumn('_run_id', F.lit(RUN)))
    # Persist to the bronze layer (as-read, all-string snapshot)
    save(df, f'{B}.{name}')
    # Record this file's metadata in the manifest
    manifest.append((RUN, name, file, int(n)))

# Save the manifest to the audit schema and display it
save(spark.createDataFrame(manifest,
    'run_id STRING, table_name STRING, file STRING, rows LONG'),
    f'{Q}.manifest')
display(spark.table(f'{Q}.manifest'))

run_id,table_name,file,rows
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,orders,olist_orders_dataset.csv,99441
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,items,olist_order_items_dataset.csv,112650
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,payments,olist_order_payments_dataset.csv,103886
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,reviews,olist_order_reviews_dataset.csv,99224
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,customers,olist_customers_dataset.csv,99441
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,products,olist_products_dataset.csv,32951
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,sellers,olist_sellers_dataset.csv,3095
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,geolocation,olist_geolocation_dataset.csv,1000163
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,translation,product_category_name_translation.csv,71


# A03 Profile blanks and exact duplicates


In [0]:
# Build a pre-cleaning profile: exact duplicate counts and null/blank counts per column
profile = []
for name in files:
    # Load the bronze (as-read) table
    raw = spark.table(f'{B}.{name}')
    # Select only data columns, excluding metadata columns prefixed with '_'
    cols = [c for c in raw.columns if not c.startswith('_')]
    df = raw.select(*cols)
    n = df.count()
    # Count exact duplicate rows (full-row duplicates across all columns)
    dup = n - df.dropDuplicates().count()
    profile.append((name, '__exact_duplicate_rows', int(dup)))
    # For each column, count rows that are NULL or contain only whitespace
    counts = df.select(*[
        F.sum(F.when(F.col(c).isNull() |
          (F.trim(F.col(c)) == ''), 1).otherwise(0)).alias(c)
        for c in cols]).first()
    # Record the blank/null count for each column
    for c in cols:
        profile.append((name, c, int(counts[c] or 0)))

# Save the full pre-cleaning profile to the audit schema
save(spark.createDataFrame(profile,
    'table_name STRING, field STRING, affected_rows LONG'),
    f'{Q}.before_profile')
# Display the profile sorted by most affected rows first
display(spark.table(f'{Q}.before_profile')
    .orderBy(F.desc('affected_rows')))
# Also show the distribution of order statuses in the bronze orders table
display(spark.table(f'{B}.orders').groupBy('order_status').count())


table_name,field,affected_rows
geolocation,__exact_duplicate_rows,261831
reviews,review_comment_title,87658
reviews,review_comment_message,58256
orders,order_delivered_customer_date,2965
orders,order_delivered_carrier_date,1783
products,product_description_lenght,610
products,product_category_name,610
products,product_photos_qty,610
products,product_name_lenght,610
orders,order_approved_at,160


order_status,count
delivered,96478
invoiced,314
shipped,1107
processing,301
unavailable,609
canceled,625
created,5
approved,2


# A04 Convert text into numbers and dates

In [0]:
# Define which columns should be cast to which data type in the silver layer

# Integer columns (counts, sequences, scores)
ints = set('''order_item_id payment_sequential payment_installments
review_score product_name_lenght product_description_lenght
product_photos_qty'''.split())
# Monetary columns (2 decimal places)
money = {'price','freight_value','payment_value'}
# Floating-point columns (measurements, coordinates)
nums = set('''product_weight_g product_length_cm product_height_cm
product_width_cm geolocation_lat geolocation_lng'''.split())
# Timestamp columns (non-timezone)
times = set('''order_purchase_timestamp order_approved_at
order_delivered_carrier_date order_delivered_customer_date
order_estimated_delivery_date shipping_limit_date
review_creation_date review_answer_timestamp'''.split())

bad = []      # Collects rows that failed type casting
balance = []  # Tracks raw vs. deduplicated row counts per table

for name in files:
    # Load bronze table and select only data columns
    raw = spark.table(f'{B}.{name}')
    cols = [c for c in raw.columns if not c.startswith('_')]
    # Drop exact duplicates before type conversion
    df = raw.select(*cols).dropDuplicates()
    # Record the row balance (raw vs. deduplicated) for audit
    balance.append((name, raw.count(), df.count()))
    for c in cols:
        # Trim whitespace and convert empty strings to NULL
        df = df.withColumn(c, F.when(F.trim(F.col(c)) == '', None)
            .otherwise(F.trim(F.col(c))))
        # Determine the target type for this column, if any
        typ = ('INT' if c in ints else 'DECIMAL(18,2)' if c in money
            else 'DOUBLE' if c in nums else
            'TIMESTAMP_NTZ' if c in times else None)
        if typ:
            # Use try_cast so uncastable values become NULL instead of failing
            x = F.expr(f'try_cast(`{c}` AS {typ})')
            # Capture rows where the original value was non-null but casting failed
            err = df.filter(F.col(c).isNotNull() & x.isNull())
            bad.append(err.select(F.lit(name).alias('table_name'),
                F.lit(c).alias('field'), F.col(c).alias('raw_value')))
            # Replace the string column with its typed version
            df = df.withColumn(c, x)
    # Save the typed, deduplicated table to the silver layer
    save(df, f'{S}.{name}')

# Union all cast-error rows into one table for audit review
save(reduce(lambda a,b: a.unionByName(b), bad), f'{Q}.cast_errors')
# Save the row-balance (raw vs. dedup) audit table
save(spark.createDataFrame(balance,
    'table_name STRING, raw_rows LONG, deduplicated_rows LONG'),
    f'{Q}.row_balance')
display(spark.table(f'{Q}.cast_errors'))
print('SILVER TYPES READY')


table_name,field,raw_value


SILVER TYPES READY


# A05 Stop if keys or core values are unusable

In [0]:
# Define the primary key columns for each table so we can check uniqueness and nulls
keys = {'orders':['order_id'], 'items':['order_id','order_item_id'],
 'payments':['order_id','payment_sequential'],
 'customers':['customer_id'], 'products':['product_id'],
 'sellers':['seller_id'], 'translation':['product_category_name']}

# Find duplicate key combinations in each table and collect them for audit
conflicts = []
for name, key in keys.items():
    df = spark.table(f'{S}.{name}')
    # Group by the key columns and keep groups with more than one row
    dup = df.groupBy(*key).count().filter('count > 1')
    # Serialize the key values to JSON for easy inspection
    conflicts.append(dup.select(F.lit(name).alias('table_name'),
        F.to_json(F.struct(*key)).alias('key_values'), 'count'))

# Union all conflict rows and save to the audit schema
save(reduce(lambda a,b: a.unionByName(b), conflicts),
    f'{Q}.key_conflicts')

# Check that no primary key column is NULL in any table (blocking)
for name, key in keys.items():
    df = spark.table(f'{S}.{name}')
    check('null_key_' + name, df.filter(
        ' OR '.join(f'`{k}` IS NULL' for k in key)).count())

# Check that there are no duplicate primary keys across all tables (blocking)
check('duplicate_keys', spark.table(f'{Q}.key_conflicts').count())

# Define business rules for required (non-null, non-negative) core values
rules = {
 'orders': 'order_purchase_timestamp IS NULL OR order_status IS NULL',
 'items': '''order_item_id < 1 OR price IS NULL OR price < 0
    OR freight_value IS NULL OR freight_value < 0''',
 'payments': '''payment_sequential < 1 OR payment_value IS NULL
    OR payment_value < 0'''}

# Run each rule as a blocking check against the silver tables
for name, rule in rules.items():
    check('required_values_' + name,
        spark.table(f'{S}.{name}').filter(rule).count())

# Validate that order_status only contains expected enum values (blocking)
allowed = ['created','approved','invoiced','processing','shipped',
           'delivered','unavailable','canceled']
check('unexpected_status', spark.table(f'{S}.orders')
    .filter(~F.col('order_status').isin(allowed)).count())

# Check for orphan items: items whose order_id has no match in the orders table (blocking)
check('orphan_items', spark.sql(f'''
 SELECT i.order_id FROM {S}.items i LEFT ANTI JOIN {S}.orders o
 ON i.order_id = o.order_id''').count())


null_key_orders 0
null_key_items 0
null_key_payments 0
null_key_customers 0
null_key_products 0
null_key_sellers 0
null_key_translation 0
duplicate_keys 0
required_values_orders 0
required_values_items 0
required_values_payments 0
unexpected_status 0
orphan_items 0


# A06 Clean optional numbers and choose one review

In [0]:
# Define optional numeric columns that may have invalid values and should be nullified
# (These are not blocking checks; invalid values are logged and set to NULL)
optional = {'products': ['product_weight_g','product_length_cm',
    'product_height_cm','product_width_cm'],   # Physical dimensions must be > 0
    'payments':['payment_installments'],            # Installment count must be > 0
    'reviews':['review_score']}                     # Score must be 1-5

exceptions = []  # Collects rows with invalid optional values for audit
for name, cols in optional.items():
    df = spark.table(f'{S}.{name}')
    for c in cols:
        # Define what 'invalid' means: score must be 1-5, all others must be > 0
        invalid = ((F.col(c) < 1) | (F.col(c) > 5)
            if c == 'review_score' else F.col(c) <= 0)
        # Log the invalid rows with table name, field, and raw value
        exceptions.append(df.filter(invalid).select(
            F.lit(name).alias('table_name'), F.lit(c).alias('field'),
            F.col(c).cast('string').alias('value')))
        # Set invalid values to NULL instead of dropping the row
        df = df.withColumn(c, F.when(invalid, None).otherwise(F.col(c)))
    # Save cleaned version to a separate _clean table (avoids read-while-overwrite issues)
    save(df, f'{S}.{name}_clean')

# Union all invalid-value rows into one audit table
save(reduce(lambda a,b: a.unionByName(b), exceptions),
    f'{Q}.optional_values')

# When an order has multiple reviews, pick one by deterministic priority ordering
review_order = Window.partitionBy('order_id').orderBy(
    F.col('review_answer_timestamp').desc_nulls_last(),  # Most recent answer first
    F.col('review_creation_date').desc_nulls_last(),     # Then most recent creation
    F.col('review_id').desc_nulls_last(),                # Then highest review_id
    F.col('review_score').desc_nulls_last(),            # Then highest score
    F.col('review_comment_title').desc_nulls_last(),    # Then presence of title
    F.col('review_comment_message').desc_nulls_last())  # Then presence of message

# Select only valid reviews (score 1-5), rank within each order, keep the top-ranked
selected = (spark.table(f'{S}.reviews_clean')
    .filter('review_score BETWEEN 1 AND 5')
    .withColumn('_rank', F.row_number().over(review_order))
    .filter('_rank = 1').select('order_id','review_score'))

# Persist the single selected review per order to the silver layer
save(selected, f'{S}.review_selected')
display(spark.table(f'{Q}.optional_values'))


table_name,field,value
products,product_weight_g,0.0
products,product_weight_g,0.0
products,product_weight_g,0.0
products,product_weight_g,0.0
payments,payment_installments,0
payments,payment_installments,0


# A07 Normalize Unicode city text safely

In [0]:
import unicodedata

# city_parts: normalize a city name into (NFC form, fold-to-ASCII key, suspect flag)
# - NFC: canonical Unicode composition for display
# - city_key: casefolded, accent-stripped, alphanumeric-only key for grouping
# - suspect: True if the name likely has encoding issues (mojibake), needs human review
def city_parts(value):
    if value is None:
        return (None, None, False)
    # Normalize to NFC (canonical composition) so decomposed forms match composed forms
    text = unicodedata.normalize('NFC', value)
    # Remove format-control characters (e.g., zero-width spaces, BOM)
    text = ''.join(c for c in text
        if unicodedata.category(c) != 'Cf')
    # Collapse internal whitespace runs and strip leading/trailing spaces
    text = ' '.join(text.split())
    if not text:
        return (None, None, False)
    # Heuristic: flag names containing replacement char or double-encoded UTF-8 patterns
    # (This is a heuristic only; human review is required for flagged names.)
    suspect = ('\ufffd' in text or re.search(
        r'[\u00c3\u00c2][\u0080-\u00bf\u201a-\u203a]', text) is not None)
    # Fold to lowercase (casefold handles special cases like German eszett)
    folded = unicodedata.normalize('NFKD', text.casefold())
    # Strip combining diacritical marks (accents) for the grouping key
    folded = ''.join(c for c in folded
        if not unicodedata.combining(c))
    # Replace non-alphanumeric chars with spaces and collapse whitespace
    key = ' '.join(''.join(c if c.isalnum() else ' '
        for c in folded).split())
    return (text, key or None, suspect)

# --- Unit tests for the city_parts function ---
assert city_parts('S\u00e3o Paulo')[1] == 'sao paulo'          # Precomposed accent
assert city_parts('Sa\u0303o Paulo')[1] == 'sao paulo'          # Decomposed accent
assert city_parts('  SAO\u00a0PAULO  ')[1] == 'sao paulo'      # Non-breaking space + padding
assert city_parts('S\u00c3O PAULO')[2] is False                # Uppercase but valid UTF-8
assert city_parts('Florian\u00f3polis')[1] == 'florianopolis'  # Accent stripping
assert city_parts('S\u00c3\u00a3o Paulo')[2] is True           # Double-encoded mojibake
assert city_parts(None)[1] is None                              # Null input

# Register as a Spark UDF returning a struct (nfc, city_key, suspect)
city_udf = F.udf(city_parts,
    'nfc STRING, city_key STRING, suspect BOOLEAN')

# Valid Brazilian state abbreviations (27 states + Federal District)
STATES = ('AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI '
          'RJ RN RS RO RR SC SP SE TO').split()
print('UNICODE TESTS PASSED')


UNICODE TESTS PASSED


# A08 Prepare customer seller and geolocation places

In [0]:
# Process customer, seller, and geolocation tables to clean state, ZIP, and city columns
places = []
for name, prefix in [('customers','customer'),('sellers','seller'),
                     ('geolocation','geolocation')]:
    df = spark.table(f'{S}.{name}')
    # Uppercase and trim the state column for validation
    state = F.upper(F.trim(F.col(prefix + '_state')))
    zipcol = F.col(prefix + '_zip_code_prefix')
    df = (df
        # Keep state only if it's a valid Brazilian state abbreviation; else NULL
        .withColumn('state_clean',
              F.when(state.isin(STATES), state))
        # Validate ZIP: must be 1-5 digits, then left-pad to 5 characters
        .withColumn('zip_clean', F.when(zipcol.rlike(r'^[0-9]{1,5}$'),
              F.lpad(zipcol, 5, '0')))
        # Apply the Unicode-normalizing city UDF and extract the struct fields
        .withColumn('_city', city_udf(F.col(prefix + '_city')))
        .withColumn('city_nfc', F.col('_city.nfc'))       # Display form (NFC)
        .withColumn('city_key', F.col('_city.city_key'))   # Grouping key (ASCII-folded)
        .withColumn('city_suspect', F.col('_city.suspect')) # Encoding issue flag
        .drop('_city'))
    # Save the cleaned place table to silver
    save(df, f'{S}.{name}_places')
    # Collect the place columns for the unified places audit table
    places.append(df.select('state_clean','city_key','city_nfc',
        'city_suspect', F.lit(name).alias('source_table')))

# Union all three place tables into one and save to audit
all_places = reduce(lambda a,b: a.unionByName(b), places)
save(all_places, f'{Q}.places')

# Find city name variants: same state + city_key but different NFC spellings
# This reveals inconsistent capitalization, accents, or encoding across sources
variants = (all_places.groupBy('state_clean','city_key')
    .agg(F.countDistinct('city_nfc').alias('spellings'),   # How many distinct spellings
         F.count('*').alias('records'),                    # Total record count
         F.collect_set('city_nfc').alias('examples')))     # All distinct spellings
save(variants, f'{Q}.city_variants')
# Display only cities with more than one spelling, sorted by record count
display(variants.filter('spellings > 1').orderBy(F.desc('records')))


state_clean,city_key,spellings,records,examples
SP,sao paulo,2,115879,"List(sao paulo, são paulo)"
DF,brasilia,2,10950,"List(brasilia, brasília)"
SP,sao bernardo do campo,2,6885,"List(sao bernardo do campo, são bernardo do campo)"
SP,santo andre,2,6562,"List(santo andre, santo andré)"
GO,goiania,2,5267,"List(goiania, goiânia)"
RJ,niteroi,2,4760,"List(niteroi, niterói)"
SC,florianopolis,2,4238,"List(florianopolis, florianópolis)"
SP,ribeirao preto,2,4236,"List(ribeirao preto, ribeirão preto)"
RJ,nova iguacu,2,4193,"List(nova iguacu, nova iguaçu)"
SP,sao jose dos campos,2,3953,"List(sao jose dos campos, são josé dos campos)"


# A09 Set reviewed city names

In [0]:
# Define human-reviewed city canonicalization rules
# Each rule maps (state, variant_key) -> (canonical_key, approved_display, reason)
# variant_key: the ASCII-folded key from city_parts that needs normalization
# canonical_key: the canonical ASCII-folded key to map to
# approved_display: the approved NFC display name to use
# reason: explanation for this mapping (for audit trail)
city_rules = [
    ('SP', 'sao paulo', 'sao paulo', 'S\u00e3o Paulo',
     'Standard display for accent and case variants')
]

# Create a Spark DataFrame from the rules list
rules = spark.createDataFrame(city_rules,
    'state_clean STRING, variant_key STRING, canonical_key STRING, '
    'approved_display STRING, reason STRING')

# Validate: no duplicate (state, variant_key) pairs (blocking)
check('city_rule_duplicate', rules.groupBy('state_clean','variant_key')
    .count().filter('count > 1').count())

# Validate: no conflicting display names for the same (state, canonical_key) (blocking)
check('city_rule_display_conflict', rules.groupBy('state_clean',
    'canonical_key').agg(F.countDistinct('approved_display').alias('n'))
    .filter('n > 1').count())

# Validate each rule: state must be valid, keys and reason must be non-empty,
# and the approved display must fold to the canonical key
for state, variant, canonical, label, reason in city_rules:
    assert state in STATES and variant and reason
    assert city_parts(label)[1] == canonical, 'Check canonical city key'

# Persist the validated rules to the silver layer
save(rules, f'{S}.city_rules')
print('CITY RULES READY')


city_rule_duplicate 0
city_rule_display_conflict 0
CITY RULES READY


# A10 Create a single city display dictionary

In [0]:
# Build a single city display dictionary that maps (state, city_key) -> display_city

# Start from all places with valid state and city_key
p = spark.table(f'{Q}.places').filter(
    'state_clean IS NOT NULL AND city_key IS NOT NULL')
# Load the human-reviewed city rules
r = spark.table(f'{S}.city_rules')

# Left join places to rules on (state_clean, variant_key)
# Keep only: non-suspect cities OR suspect cities that have an approved display
j = (p.alias('p').join(r.alias('r'),
    (F.col('p.state_clean') == F.col('r.state_clean')) &
    (F.col('p.city_key') == F.col('r.variant_key')), 'left')
    .select('p.*', 'r.canonical_key', 'r.approved_display')
    # Exclude suspect cities without a rule (they need human review)
    .filter('NOT city_suspect OR approved_display IS NOT NULL')
    # final_key = canonical_key if a rule matched, otherwise the original city_key
    .withColumn('final_key', F.coalesce('canonical_key','city_key')))

# Group by (state, final_key, nfc_spelling) to count how often each spelling appears
labels = j.groupBy('state_clean','final_key','city_nfc').count()
# Pick the most frequent NFC spelling per (state, final_key) as the default display
w = Window.partitionBy('state_clean','final_key').orderBy(
    F.desc('count'), F.asc('city_nfc'))
labels = labels.withColumn('rn', F.row_number().over(w)).filter('rn=1')

# Get approved display names from the rules (override the most-frequent heuristic)
approved = r.select('state_clean', F.col('canonical_key').alias('final_key'),
    'approved_display').dropDuplicates()

# Build the final dictionary: join labels with approved overrides
dictionary = (labels.join(approved, ['state_clean','final_key'], 'left')
    # Use approved display if available, else fall back to the most-frequent NFC spelling
    .withColumn('display_city', F.coalesce('approved_display','city_nfc'))
    # Create a unique city_id as 'state|final_key'
    .withColumn('city_id', F.concat_ws('|','state_clean','final_key'))
    .select('state_clean','final_key','city_id','display_city'))

# Validate: no duplicate city_ids in the dictionary (blocking)
check('city_dictionary_duplicate', dictionary.groupBy('city_id')
    .count().filter('count > 1').count())
save(dictionary, f'{S}.city_dictionary')
print('CITY DICTIONARY READY')


city_dictionary_duplicate 0
CITY DICTIONARY READY


# A11 Apply city cleaning and collapse geographic duplicates

In [0]:
# Apply city canonicalization to customers and sellers tables
for name in ['customers','sellers']:
    p = spark.table(f'{S}.{name}_places')
    r = spark.table(f'{S}.city_rules')
    # Left join places to city rules on (state_clean, variant_key)
    j = p.alias('p').join(r.alias('r'),
        (F.col('p.state_clean') == F.col('r.state_clean')) &
        (F.col('p.city_key') == F.col('r.variant_key')), 'left')
    # Compute final_key only for non-suspect cities or suspect cities with an approved rule
    j = (j.select('p.*','r.canonical_key','r.approved_display')
        .withColumn('final_key', F.when(
            (~F.col('city_suspect')) | F.col('approved_display').isNotNull(),
            F.coalesce('canonical_key','city_key'))))
    # Join to the city dictionary to get display_city and city_id
    d = spark.table(f'{S}.city_dictionary')
    j = j.join(d, ['state_clean','final_key'], 'left')
    # Fill missing display/city_id with 'Unknown' defaults
    j = j.withColumn('display_city', F.coalesce('display_city',
        F.lit('Unknown'))).withColumn('city_id', F.coalesce('city_id',
        F.concat_ws('|', F.coalesce('state_clean',F.lit('Unknown')),
            F.lit('unknown'))))
    # Validate: row count should not change after the join (blocking)
    check(name + '_city_join_rows', abs(j.count() - p.count()))
    # Persist the cleaned customer/seller table to silver
    save(j, f'{S}.{name}_clean')

# --- Geolocation: collapse to one lookup row per ZIP code ---
geo = spark.table(f'{S}.geolocation_places')
# Keep only rows with valid ZIP, state, and plausible coordinates
valid_geo = geo.filter('''zip_clean IS NOT NULL AND state_clean IS NOT NULL
 AND geolocation_lat BETWEEN -90 AND 90
 AND geolocation_lng BETWEEN -180 AND 180''')
# Group by ZIP code and aggregate:
# - state_count: how many distinct states claim this ZIP (detect ambiguity)
# - state: first state seen (used when unambiguous)
# - latitude/longitude: median coordinates across all rows for this ZIP
lookup = valid_geo.groupBy('zip_clean').agg(
    F.countDistinct('state_clean').alias('state_count'),
    F.first('state_clean').alias('state'),
    F.expr('percentile_approx(geolocation_lat,0.5)').alias('latitude'),
    F.expr('percentile_approx(geolocation_lng,0.5)').alias('longitude'))
# Save unambiguous ZIP lookups (single state) to silver
save(lookup.filter('state_count = 1'), f'{S}.geolocation_zip')
# Save ambiguous ZIP lookups (multiple states) to audit for review
save(lookup.filter('state_count > 1'), f'{Q}.ambiguous_zip')
print('CITY CLEANING APPLIED')


customers_city_join_rows 0
sellers_city_join_rows 0
CITY CLEANING APPLIED


# A12 Create the reporting dimensions

In [0]:
# --- Create the reporting dimension tables in the gold layer ---

# dim_customer: one row per customer_id with cleaned state, ZIP, and city info
spark.sql(f'''CREATE OR REPLACE TABLE {G}.dim_customer AS
 SELECT customer_id, customer_unique_id,
   COALESCE(state_clean,'Unknown') AS customer_state,
   zip_clean, city_id, display_city
 FROM {S}.customers_clean''')

# dim_seller: one row per seller_id with cleaned state, ZIP, and city info
spark.sql(f'''CREATE OR REPLACE TABLE {G}.dim_seller AS
 SELECT seller_id, COALESCE(state_clean,'Unknown') AS seller_state,
   zip_clean, city_id, display_city FROM {S}.sellers_clean''')

# dim_product: one row per product with English category name (left joined from translation)
spark.sql(f'''CREATE OR REPLACE TABLE {G}.dim_product AS
 SELECT p.*, COALESCE(t.product_category_name_english,
   'Unknown') AS category
 FROM {S}.products_clean p LEFT JOIN {S}.translation t
 ON p.product_category_name = t.product_category_name''')

# dim_date: a calendar dimension spanning the min and max purchase dates
spark.sql(f'''CREATE OR REPLACE TABLE {G}.dim_date AS
 WITH bounds AS (
   -- Find the earliest and latest purchase dates across all orders
   SELECT MIN(CAST(order_purchase_timestamp AS DATE)) AS lo,
          MAX(CAST(order_purchase_timestamp AS DATE)) AS hi
   FROM {S}.orders
 ), dates AS (
   -- Generate one row per day between the bounds
   SELECT EXPLODE(SEQUENCE(lo,hi,INTERVAL 1 DAY)) AS calendar_date
   FROM bounds
 )
 -- Extract useful date components for reporting
 SELECT calendar_date, YEAR(calendar_date) AS year,
   MONTH(calendar_date) AS month_number,
   CAST(DATE_TRUNC('MONTH',calendar_date) AS DATE) AS month_start,
   QUARTER(calendar_date) AS quarter
 FROM dates''')
print('DIMENSIONS READY')


DIMENSIONS READY


# A13 Create one reporting row per order

In [0]:
# Create the fact_order table: one reporting row per order with aggregated items, payments, review, and customer info
spark.sql(f'''CREATE OR REPLACE TABLE {G}.fact_order AS
 -- Aggregate item-level data per order: item count, total price, total freight
 WITH i AS (
   SELECT order_id, COUNT(*) AS item_count, SUM(price) AS item_value,
     SUM(freight_value) AS freight FROM {S}.items GROUP BY order_id
 ),
 -- Aggregate payment values per order
 p AS (
   SELECT order_id, SUM(payment_value) AS payment_value
   FROM {S}.payments_clean GROUP BY order_id
 )
 SELECT o.*,  -- All columns from the silver orders table
   -- Extract purchase date for joining to dim_date
   CAST(o.order_purchase_timestamp AS DATE) AS purchase_date,
   -- Customer dimension columns (with defaults for missing joins)
   c.customer_unique_id,
   COALESCE(c.customer_state,'Unknown') AS customer_state,
   COALESCE(c.city_id,'Unknown|unknown') AS city_id,
   COALESCE(c.display_city,'Unknown') AS display_city,
   -- Item and payment aggregates (item_count defaults to 0 if no items)
   COALESCE(i.item_count,0) AS item_count,
   i.item_value, i.freight, p.payment_value,
   -- Selected review score (one per order)
   r.review_score,
   -- Delivery days: only computed for delivered orders with valid delivery date
   CASE WHEN o.order_status='delivered'
     AND o.order_delivered_customer_date >= o.order_purchase_timestamp
   THEN DATEDIFF(CAST(o.order_delivered_customer_date AS DATE),
                 CAST(o.order_purchase_timestamp AS DATE))
   END AS delivery_days
 FROM {S}.orders o
 -- Left joins so all orders are kept even if items/payments/reviews are missing
 LEFT JOIN i ON o.order_id=i.order_id
 LEFT JOIN p ON o.order_id=p.order_id
 LEFT JOIN {S}.review_selected r ON o.order_id=r.order_id
 LEFT JOIN {G}.dim_customer c ON o.customer_id=c.customer_id''')
print('ORDER FACT READY')


ORDER FACT READY


# A14 Create dashboard measures and the item table

In [0]:
# --- Create BI-ready views and the item-level fact table ---

# bi_orders: a convenience view over fact_order with pre-computed dashboard measures
spark.sql(f'''CREATE OR REPLACE VIEW {G}.bi_orders AS
 SELECT *,  -- All columns from fact_order
   1 AS order_count,  -- Constant 1 for COUNT-like aggregation
   -- Flag: 1 if the order was delivered, 0 otherwise
   CASE WHEN order_status='delivered' THEN 1 ELSE 0 END
     AS delivered_order,
   -- GMV: item_value only for delivered orders (NULL otherwise, so SUM ignores non-delivered)
   CASE WHEN order_status='delivered' THEN item_value END
     AS delivered_gmv,
   -- On-time flag: 1 if delivered on or before estimated date, 0 if late, NULL if not applicable
   CASE WHEN delivery_days IS NOT NULL
     AND order_estimated_delivery_date >= order_purchase_timestamp
   THEN CASE WHEN CAST(order_delivered_customer_date AS DATE) <=
     CAST(order_estimated_delivery_date AS DATE) THEN 1 ELSE 0 END
   END AS on_time_flag
 FROM {G}.fact_order''')

# fact_order_item: item-level fact table enriched with order-level and product attributes
spark.sql(f'''CREATE OR REPLACE TABLE {G}.fact_order_item AS
 SELECT i.*,  -- All item-level columns from silver
   -- Order-level attributes joined from the fact table
   o.purchase_date, o.order_status, o.customer_state,
   o.city_id, o.display_city,
   -- Product category from dim_product (defaults to 'Unknown')
   COALESCE(p.category,'Unknown') AS category
 FROM {S}.items i JOIN {G}.fact_order o ON i.order_id=o.order_id
 LEFT JOIN {G}.dim_product p ON i.product_id=p.product_id''')

# bi_items: delivered-only item view for GMV and category analysis
spark.sql(f'''CREATE OR REPLACE VIEW {G}.bi_items AS
 SELECT * FROM {G}.fact_order_item WHERE order_status='delivered' ''')
print('BI TABLES READY')


BI TABLES READY


# A15 Validate counts and money

In [0]:
# --- Validate that the gold layer preserves counts and money totals from silver ---

# Load the BI views/tables for validation
orders = spark.table(f'{G}.bi_orders')
items = spark.table(f'{G}.fact_order_item')

# Check: order count in gold matches silver orders (blocking)
check('order_count', abs(orders.count() - spark.table(f'{S}.orders').count()))
# Check: item count in gold matches silver items (blocking)
check('item_count', abs(items.count() - spark.table(f'{S}.items').count()))
# Check: no duplicate order_ids in the BI orders view (blocking)
check('order_unique', orders.groupBy('order_id').count()
    .filter('count > 1').count())
# Check: no duplicate (order_id, order_item_id) pairs in the item fact (blocking)
check('item_unique', items.groupBy('order_id','order_item_id').count()
    .filter('count > 1').count())
# Check: every delivered order has at least one item with a price (blocking)
check('delivered_missing_items', orders.filter(
    "order_status='delivered' AND (item_count=0 OR item_value IS NULL)").count())

# Check: total price and total freight in gold match silver (blocking)
for field in ['price','freight_value']:
    a = spark.table(f'{S}.items').agg(F.sum(field)).first()[0]  # Silver total
    b = items.agg(F.sum(field)).first()[0]                       # Gold total
    check('total_' + field, int(a != b))

# Check: on_time_flag only contains 0, 1, or NULL (blocking)
check('invalid_on_time', orders.filter(
    'on_time_flag IS NOT NULL AND on_time_flag NOT IN (0,1)').count())

# Diagnose payment vs. item+freight mismatches (non-blocking; saved for review)
diagnostic = spark.sql(f'''
 SELECT order_id, order_status, item_value, freight, payment_value,
   payment_value - (item_value+freight) AS difference
 FROM {G}.fact_order
 WHERE payment_value IS NULL OR item_value IS NULL
   OR ABS(payment_value-(item_value+freight))>0.01''')
save(diagnostic, f'{Q}.payment_differences')
print('BLOCKING VALIDATION PASSED')


order_count 0
item_count 0
order_unique 0
item_unique 0
delivered_missing_items 0
total_price 0
total_freight_value 0
invalid_on_time 0
BLOCKING VALIDATION PASSED


# A16 Save warnings and the control totals

In [0]:
# Define non-blocking warning queries: data quality issues to review but not halt the pipeline
# Each query finds rows with potential problems; results are saved to audit and logged as non-blocking checks
# Clarence: PLEASE DON'T CHANGE THESE QUERIES. Check with me first!
warning_sql = {
 # Orders whose customer_id has no match in the customers table
 'orders_missing_customer': f'''SELECT o.* FROM {S}.orders o
 LEFT ANTI JOIN {S}.customers c ON o.customer_id=c.customer_id''',
 # Items whose product_id has no match in the products table
 'items_missing_product': f'''SELECT i.* FROM {S}.items i
 LEFT ANTI JOIN {S}.products p ON i.product_id=p.product_id''',
 # Items whose seller_id has no match in the sellers table
 'items_missing_seller': f'''SELECT i.* FROM {S}.items i
 LEFT ANTI JOIN {S}.sellers s ON i.seller_id=s.seller_id''',
 # Cities flagged as suspect (possible encoding issues) for human review
 'city_encoding_review': f'''SELECT * FROM {Q}.places
 WHERE city_suspect = true''',
 # Delivered orders where on_time_flag could not be computed
 'delivery_excluded': f'''SELECT * FROM {G}.bi_orders
 WHERE order_status='delivered' AND on_time_flag IS NULL''',
 # Customers with missing state, ZIP, or unknown city
 'customer_geo_incomplete': f'''SELECT * FROM {S}.customers_clean
 WHERE state_clean IS NULL OR zip_clean IS NULL OR display_city='Unknown' ''',
 # Products with no English category translation
 'category_unknown': f'''SELECT * FROM {G}.dim_product
 WHERE category='Unknown' ''',
 # Geolocation rows with missing or out-of-range coordinates
 'invalid_coordinates': f'''SELECT * FROM {S}.geolocation_places
 WHERE geolocation_lat IS NULL OR geolocation_lng IS NULL
 OR NOT (geolocation_lat BETWEEN -90 AND 90)
 OR NOT (geolocation_lng BETWEEN -180 AND 180)'''}

# Execute each warning query, save results to audit, and log as non-blocking checks
for name, query in warning_sql.items():
    df = spark.sql(query)
    save(df, f'{Q}.{name}')
    check(name, df.count(), False)  # blocking=False so pipeline continues

# Compute baseline KPIs for the entire dataset (one summary row)
baseline = spark.sql(f'''SELECT COUNT(*) AS orders,
 SUM(delivered_order) AS delivered_orders,
 SUM(delivered_gmv) AS delivered_gmv,
 AVG(delivered_gmv) AS delivered_aov,
 AVG(on_time_flag) AS on_time_rate,
 COUNT(on_time_flag) AS eligible_deliveries,
 AVG(review_score) AS mean_review,
 COUNT(review_score) AS reviewed_orders,
 MIN(purchase_date) AS first_date, MAX(purchase_date) AS last_date
 FROM {G}.bi_orders''')
# Save baseline KPIs to audit and display them
save(baseline, f'{Q}.baseline')
display(baseline)
# Display the full QA checks log (blocking and non-blocking)
display(spark.table(f'{Q}.checks'))


orders_missing_customer 0
items_missing_product 0
items_missing_seller 0
city_encoding_review 0
delivery_excluded 8
customer_geo_incomplete 0
category_unknown 623
invalid_coordinates 0


orders,delivered_orders,delivered_gmv,delivered_aov,on_time_rate,eligible_deliveries,mean_review,reviewed_orders,first_date,last_date
99441,96478,13221498.11,137.041586,0.9322690992018244,96470,4.086386346822333,98673,2016-09-04,2018-10-17


run_id,test,failures,blocking
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_orders,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_items,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_payments,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_customers,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_products,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_sellers,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,null_key_translation,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,duplicate_keys,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,required_values_orders,0,true
b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba,required_values_items,0,true


# A17 Export two gold public datasets

In [0]:
# --- Export two small public datasets as CSV files ---

# Safety gate: ensure no blocking QA checks failed before exporting
assert not any(block and count for _,_,count,block in qa)
# Gate: confirm that Section 8 (warnings review) has been completed
WARNINGS_REVIEWED = True  # Set True after the Section 8 review.
assert WARNINGS_REVIEWED, 'Complete Section 8 before exporting'

# Define which columns to export from each BI view
exports = {
 'orders': '''order_id,purchase_date,customer_state,city_id,display_city,
 order_count,delivered_order,delivered_gmv,delivery_days,
 on_time_flag,review_score''',
 'items': '''order_id,order_item_id,purchase_date,customer_state,
 city_id,display_city,category,price,freight_value'''}

# Create a unique export directory for this pipeline run
EXPORT = f'{BASE}/exports/{RUN}'
for name, columns in exports.items():
    # Select the specified columns from the BI view
    df = spark.sql(f'SELECT {columns} FROM {G}.bi_{name}')
    # Write as a single CSV file (coalesce(1)) with header, UTF-8, formatted dates, empty nulls
    (df.coalesce(1).write.mode('overwrite').option('header',True)
        .option('encoding','UTF-8').option('dateFormat','yyyy-MM-dd')
        .option('nullValue','').csv(f'{EXPORT}/{name}'))
    print(name, df.count(), f'{EXPORT}/{name}')
print('EXPORT READY', EXPORT)


orders 99441 /Volumes/workspace/olist_raw11/files/exports/b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba/orders
items 110197 /Volumes/workspace/olist_raw11/files/exports/b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba/items
EXPORT READY /Volumes/workspace/olist_raw11/files/exports/b86c5ec5-6d1e-4b09-9b19-e34cb6eb8aba


# A18 Explore trends and city cleaning results

In [0]:
# --- Exploratory queries: monthly trends, state performance, and city cleaning results ---

# Query 1: Monthly order volume and delivered GMV trend over time
display(spark.sql(f'''SELECT
 CAST(DATE_TRUNC('MONTH',purchase_date) AS DATE) AS month,
 COUNT(*) AS orders, SUM(delivered_gmv) AS delivered_gmv
 FROM {G}.bi_orders GROUP BY 1 ORDER BY 1'''))

# Query 2: Per-state delivery performance (on-time rate, mean delivery days, reviews)
# Only shows states with at least 30 eligible deliveries for statistical reliability
display(spark.sql(f'''SELECT customer_state,
 COUNT(on_time_flag) AS eligible, AVG(on_time_flag) AS on_time_rate,
 AVG(delivery_days) AS mean_days, COUNT(review_score) AS reviewed,
 AVG(review_score) AS mean_review
 FROM {G}.bi_orders GROUP BY customer_state
 HAVING COUNT(on_time_flag)>=30 ORDER BY on_time_rate'''))

# Query 3: Verify city cleaning results for São Paulo, SP
# Shows how the original city variants map to the canonical display_city
display(spark.sql(f'''SELECT customer_city AS original_city,
 city_nfc, city_key, city_id, display_city, COUNT(*) AS records
 FROM {S}.customers_clean
 WHERE state_clean='SP' AND final_key='sao paulo'
 GROUP BY 1,2,3,4,5 ORDER BY records DESC'''))


month,orders,delivered_gmv
2016-09-01,4,134.97
2016-10-01,324,40325.11
2016-12-01,1,10.90
2017-01-01,800,111798.36
2017-02-01,1780,234223.40
2017-03-01,2682,359198.85
2017-04-01,2404,340669.68
2017-05-01,3700,489338.25
2017-06-01,3245,421923.37
2017-07-01,4026,481604.52


customer_state,eligible,on_time_rate,mean_days,reviewed,mean_review
AL,397,0.7858942065491183,24.50125944584383,410,3.7560975609756095
MA,717,0.8256624825662483,21.511854951185494,742,3.757412398921833
SE,335,0.8477611940298507,21.46268656716418,349,3.8080229226361033
PI,476,0.8613445378151261,19.394957983193276,490,3.9183673469387754
CE,1279,0.8623924941360438,21.200156372165754,1326,3.856711915535445
RR,41,0.8780487804878049,29.341463414634145,46,3.608695652173913
BA,3256,0.8783783783783784,19.278562653562652,3340,3.8616766467065866
RJ,12350,0.8789473684210526,15.237004048582996,12687,3.877433593442106
PA,946,0.8879492600422833,23.725158562367866,962,3.8503118503118503
ES,1995,0.8927318295739348,15.723809523809523,2006,4.037387836490528


original_city,city_nfc,city_key,city_id,display_city,records
sao paulo,sao paulo,sao paulo,SP|sao paulo,São Paulo,15540


# A19 Test money joins and delivery rules

In [0]:
# --- Unit tests for the money-join and on-time delivery business rules ---

# Test 1: Verify that SUM(price) and SUM(payment) correctly aggregate per order
# Two items (10+20=30) and two payments (15+15=30) for order 'o1' should both equal 30
test = spark.sql('''WITH
 items AS (SELECT * FROM VALUES ('o1',10),('o1',20) AS t(id,price)),
 payments AS (SELECT * FROM VALUES ('o1',15),('o1',15) AS t(id,amount)),
 i AS (SELECT id,SUM(price) AS value FROM items GROUP BY id),
 p AS (SELECT id,SUM(amount) AS paid FROM payments GROUP BY id)
 SELECT i.id,i.value,p.paid FROM i JOIN p ON i.id=p.id''').collect()
assert len(test)==1 and test[0].value==30 and test[0].paid==30

# Test 2: Verify the on_time_flag CASE expression against four scenarios:
# - same_day: delivered before estimated -> flag=1 (on time)
# - late: delivered after estimated -> flag=0 (late)
# - missing: no delivery date -> flag=NULL (not applicable)
# - impossible: delivery before purchase (bad data) -> flag=NULL (excluded)
dates = spark.sql('''WITH t AS (
 SELECT * FROM VALUES
 ('same_day','2018-01-01 10:00:00','2018-01-05 20:00:00',
  '2018-01-05 00:00:00'),
 ('late','2018-01-01 10:00:00','2018-01-06 08:00:00',
  '2018-01-05 00:00:00'),
 ('missing','2018-01-01 10:00:00','2018-01-05 08:00:00',NULL),
 ('impossible','2018-01-03 10:00:00','2018-01-02 08:00:00',
  '2018-01-05 00:00:00') AS t(id,purchase,delivered,estimated)
 )
 SELECT id, CASE
 WHEN CAST(delivered AS TIMESTAMP_NTZ)>=CAST(purchase AS TIMESTAMP_NTZ)
  AND CAST(estimated AS TIMESTAMP_NTZ)>=CAST(purchase AS TIMESTAMP_NTZ)
 THEN CASE WHEN CAST(delivered AS DATE)<=CAST(estimated AS DATE)
           THEN 1 ELSE 0 END END AS flag
 FROM t''').collect()
observed = {r.id:r.flag for r in dates}
assert observed == {'same_day':1,'late':0,
                    'missing':None,'impossible':None}
print('BUSINESS RULE FIXTURES PASSED')


BUSINESS RULE FIXTURES PASSED


# A20 Service Analysis (EXTRA TEST, NOT RECORDED IN OUR DOCUMENTATION, read-only)

In [0]:
# Order-level: review score for on-time vs late deliveries
display(spark.sql(f'''SELECT CASE WHEN on_time_flag=1 THEN 'on time' ELSE 'late' END AS delivery,
  COUNT(review_score) AS reviewed, AVG(review_score) AS mean_review
  FROM {G}.bi_orders WHERE on_time_flag IS NOT NULL GROUP BY 1'''))

# RJ deliveries by seller state (orders with several sellers appear once per seller state)
display(spark.sql(f'''WITH lane AS (
  SELECT DISTINCT o.order_id, s.seller_state, o.on_time_flag
  FROM {G}.bi_orders o JOIN {G}.fact_order_item i ON o.order_id=i.order_id
  JOIN {G}.dim_seller s ON i.seller_id=s.seller_id
  WHERE o.customer_state='RJ' AND o.on_time_flag IS NOT NULL)
  SELECT seller_state, COUNT(*) AS orders, AVG(on_time_flag) AS on_time_rate
  FROM lane GROUP BY 1 HAVING COUNT(*)>=30 ORDER BY on_time_rate'''))

delivery,reviewed,mean_review
on time,89443,4.290386055923885
late,6381,2.270490518727472


seller_state,orders,on_time_rate
SP,8188,0.8593063019052272
PR,978,0.8803680981595092
SC,474,0.9092827004219409
ES,59,0.9152542372881356
MG,1145,0.9179039301310044
DF,96,0.9270833333333334
PE,42,0.9285714285714286
RS,246,0.9308943089430894
GO,56,0.9464285714285714
MA,41,0.9512195121951219
